In [ ]:
import pandas as pd
import os

# 1. 기본 경로 설정
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
path_score = os.path.join(base_path, "output", "PROD_FLOWSCORE", "PUBLIC")
path_point = os.path.join(base_path, "output", "PROD_FLOWPOINT", "PUBLIC")

print("🚚 원본 데이터 로딩을 시작합니다...")

try:
    # 핵심 11개 데이터셋 로드
    ov = pd.read_csv(os.path.join(path_score, "COMPANY_OVERVIEW.csv"), encoding='utf-8-sig')
    bs = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_STATEMENT.csv"), encoding='utf-8-sig')
    is_df = pd.read_csv(os.path.join(path_score, "COMPANY_INCOME_STATEMENT.csv"), encoding='utf-8-sig')
    df_ratio_raw = pd.read_csv(os.path.join(path_score, "COMPANY_FINANCIAL_RATIO.csv"), encoding='utf-8-sig')
    cg = pd.read_csv(os.path.join(path_score, "COMPANY_CREDIT_GRADE.csv"), encoding='utf-8-sig')
    od = pd.read_csv(os.path.join(path_score, "COMPANY_OVERDUE.csv"), encoding='utf-8-sig')
    emp = pd.read_csv(os.path.join(path_score, "COMPANY_EMPLOYEE_STATISTICS.csv"), encoding='utf-8-sig')
    rep = pd.read_csv(os.path.join(path_score, "COMPANY_REPRESENTATIVE_HISTORY.csv"), encoding='utf-8-sig')
    cap = pd.read_csv(os.path.join(path_score, "COMPANY_CAPITAL_STATUS.csv"), encoding='utf-8-sig')
    link = pd.read_csv(os.path.join(path_point, "COMPANY_LINKS.csv"), encoding='utf-8-sig')
    bnpl = pd.read_csv(os.path.join(path_point, "VIEW_PAY_BNPL_REQUEST_HISTORY.csv"), encoding='utf-8-sig')
    
    print("✅ 모든 파일이 메모리에 성공적으로 로드되었습니다.")
    print(f"📍 로드된 재무비율 데이터 수: {len(df_ratio_raw)}건")
except Exception as e:
    print(f"❌ 데이터 로드 중 오류 발생: {e}")

🚚 원본 데이터 로딩을 시작합니다...
✅ 모든 파일이 메모리에 성공적으로 로드되었습니다.
📍 로드된 재무비율 데이터 수: 3580건


In [5]:
import pandas as pd

def audit_raw_files(file_dict):
    audit_results = []
    
    print("🔍 [276 Data Audit] 원본 파일 전수 조사 시작...")
    print("-" * 70)
    
    for name, df in file_dict.items():
        if df is None:
            audit_results.append({'파일': name, '상태': '❌ 로드 실패', '행수': 0, '결측치': 0})
            continue
            
        # 기본 정보
        rows, cols = df.shape
        null_count = df.isnull().sum().sum()
        
        # 핵심 컬럼 존재 여부 체크 (라벨링 전이므로 원본 컬럼명 기준)
        has_id = 'COMPANY_ID' in df.columns or 'ID' in df.columns
        
        # 특정 파일별 핵심 지표 체크
        extra_check = ""
        if name == 'ratio':
            # 이자보상배수(FR_VAL8 등)가 있는지 샘플 확인
            has_interest = any('FR_VAL' in c for c in df.columns)
            extra_check = "재무지표포함" if has_interest else "지표부족"
            
        audit_results.append({
            '파일': name,
            '행수': rows,
            '컬럼수': cols,
            'ID보유': '✅' if has_id else '❌',
            '전체결측치': null_count,
            '비고': extra_check
        })

    audit_df = pd.DataFrame(audit_results)
    display(audit_df)
    return audit_df

# 1. 검수 대상 리스트업 (현재 로드된 변수명 기준)
files_to_check = {
    'ov': ov, 'bs': bs, 'is': is_df, 'ratio': df_ratio_raw,
    'cg': cg, 'od': od, 'emp': emp, 'rep': rep, 
    'cap': cap, 'link': link, 'bnpl': bnpl
}

# 2. 진단 실행
df_audit_report = audit_raw_files(files_to_check)

🔍 [276 Data Audit] 원본 파일 전수 조사 시작...
----------------------------------------------------------------------


,파일,행수,컬럼수,ID보유,전체결측치,비고
0,ov,1753,42,✅,13551,
1,bs,3562,392,✅,1198089,
2,is,3561,101,✅,214763,
3,ratio,3580,27,✅,11782,재무지표포함
4,cg,9461,16,✅,11038,
5,od,34,25,✅,182,
6,emp,7881,12,✅,7881,
7,rep,1735,11,✅,1735,
8,cap,1568,20,✅,9767,
9,link,25036,18,✅,50072,


In [7]:
# 3대 비재무 파일의 실제 컬럼명 리스트 확인
print("📋 [수색 결과] 실제 컬럼명을 확인합니다.")
print("-" * 50)

if 'rep' in globals():
    print(f"👤 대표자(rep) 컬럼: {rep.columns.tolist()}")

if 'emp' in globals():
    print(f"\n👥 인력(emp) 컬럼: {emp.columns.tolist()}")

if 'link' in globals():
    print(f"\n🔗 링크(link) 컬럼: {link.columns.tolist()}")

📋 [수색 결과] 실제 컬럼명을 확인합니다.
--------------------------------------------------
👤 대표자(rep) 컬럼: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', 'CREATED_AT', '_AB_CDC_LSN', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'REPRESENTATIVE_HISTORY_INFO']

👥 인력(emp) 컬럼: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'COMPANY_ID', 'CREATED_AT', '_AB_CDC_LSN', 'STANDARD_DATE', 'EMPLOYEE_COUNT', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT']

🔗 링크(link) 컬럼: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'AMOUNT', 'CATEGORY', 'COMPANY_ID', 'CREATED_AT', 'DELETED_AT', 'TRADE_DATE', 'UPDATED_AT', '_AB_CDC_LSN', 'REFERENCE_ID', 'REFERENCE_TABLE', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'ANOTHER_COMPANY_ID']


In [8]:
import pandas as pd

def deep_check_v2(rep, emp, link):
    print("📋 [276 Data Factory] 비재무 원본 데이터 내용물 정밀 점검")
    print("="*60)
    
    # 1. 대표자 이력 점검 (정보가 텍스트로 뭉쳐 있는지 확인)
    print("\n👤 [대표자 이력] 정보 추출 상태 점검")
    # 'REPRESENTATIVE_HISTORY_INFO'에 이름이나 직함이 들어있는지 확인
    display(rep[['COMPANY_ID', 'REPRESENTATIVE_HISTORY_INFO', 'CREATED_AT']].head(3))
    
    # 2. 인력 통계 점검 (수치 범위 및 결측치)
    print("\n👥 [인력 통계] 고용 지표 분포")
    # EMPLOYEE_COUNT가 숫자인지, 상식적인 범위인지 확인
    emp['EMPLOYEE_COUNT'] = pd.to_numeric(emp['EMPLOYEE_COUNT'], errors='coerce')
    display(emp[['COMPANY_ID', 'STANDARD_DATE', 'EMPLOYEE_COUNT']].describe().T)
    
    # 3. 네트워크 링크 점검 (금액 및 거래처 확인)
    print("\n🔗 [네트워크 링크] 거래 관계 및 규모")
    # AMOUNT 필드가 거래 규모를 나타내는지 확인
    display(link[['COMPANY_ID', 'ANOTHER_COMPANY_ID', 'AMOUNT', 'CATEGORY']].head(3))

# 실행
deep_check_v2(rep, emp, link)

📋 [276 Data Factory] 비재무 원본 데이터 내용물 정밀 점검

👤 [대표자 이력] 정보 추출 상태 점검


,COMPANY_ID,REPRESENTATIVE_HISTORY_INFO,CREATED_AT
0,1,"[{""name"": ""박진완"", ""title"": ""사내이사"", ""sequence"": ...",2024-12-17 09:29:15.641610
1,7,"[{""name"": ""김유훈"", ""title"": ""대표이사"", ""sequence"": ...",2024-12-20 07:18:40.674974
2,13,"[{""name"": ""김한준"", ""title"": ""대표이사"", ""sequence"": ...",2024-12-20 07:52:08.386839



👥 [인력 통계] 고용 지표 분포


,count,mean,std,min,25%,50%,75%,max
COMPANY_ID,7881.0,796.348560,482.684884,1.0,365.0,832.0,1247.0,1598.0
EMPLOYEE_COUNT,7881.0,1436.785687,9417.969336,0.0,9.0,30.0,125.0,129524.0



🔗 [네트워크 링크] 거래 관계 및 규모


,COMPANY_ID,ANOTHER_COMPANY_ID,AMOUNT,CATEGORY
0,95,94,500000000,receivable
1,366,94,500000000,receivable
2,400,399,16444642,receivable


In [18]:
import pandas as pd

def safe_merge_factory(ov, ratio_snap, rep_score, emp_latest):
    print("🔍 [Step 3] 기업명 컬럼 탐색 및 최종 병합 시작...")
    
    # 1. 기업명 후보군 중 실제 존재하는 컬럼 찾기
    name_candidates = ['CPN_NM', 'NM', 'COMPANY_NAME', 'CPN_NAME', '기업명']
    actual_name_col = next((col for col in name_candidates if col in ov.columns), None)
    
    if not actual_name_col:
        print(f"⚠️ 경고: 기업명 컬럼을 찾을 수 없습니다. 현재 컬럼: {ov.columns.tolist()[:5]}")
        # 컬럼이 정 안 보이면 첫 번째 컬럼이라도 가져오거나, 에러 방지를 위해 빈 컬럼 생성
        actual_name_col = 'CPN_NM'
        ov[actual_name_col] = "Unknown"
    else:
        print(f"✅ 확인된 기업명 컬럼: '{actual_name_col}'")

    # 2. 최종 병합 실행 (KeyError 방지)
    # ov에서 '회사 아이디'와 위에서 찾은 'actual_name_col'만 추출
    try:
        master = pd.merge(
            ov[['회사 아이디', actual_name_col]], 
            ratio_snap[['회사 아이디', '부채비율', '영업이익율', '이자보상배수']], 
            on='회사 아이디', 
            how='left'
        )
        # 기업명 컬럼명을 표준인 '기업명'으로 통일
        master = master.rename(columns={actual_name_col: '기업명'})
        
        # 나머지 병합 진행
        master = pd.merge(master, rep_score, on='회사 아이디', how='left')
        master = pd.merge(master, emp_latest[['회사 아이디', '종업원수']], on='회사 아이디', how='left')
        
        print(f"✅ 최종 마트 구성 완료! (총 {len(master)}개 기업)")
        return master
        
    except Exception as e:
        print(f"❌ 병합 중 치명적 오류 발생: {e}")
        # 에러 발생 시 현재 ov의 컬럼명을 출력해서 확인하게 함
        print(f"현재 ov 컬럼 리스트: {ov.columns.tolist()}")
        return pd.DataFrame()

# 실행
df_final_100 = safe_merge_factory(ov, ratio_snap, rep_score, emp_latest)

# 결과 확인
if not df_final_100.empty:
    display(df_final_100.head(10))

🔍 [Step 3] 기업명 컬럼 탐색 및 최종 병합 시작...
✅ 확인된 기업명 컬럼: 'COMPANY_NAME'
✅ 최종 마트 구성 완료! (총 1753개 기업)


,회사 아이디,기업명,부채비율,영업이익율,이자보상배수,변경횟수,종업원수
0,0000000001,청춘에프앤비,+88.87,0.43,443.3,1.0,10.0
1,0000000007,일신태광금속,NaN,NaN,NaN,1.0,18.0
2,0000000013,아름일렉트로닉스,NaN,NaN,NaN,1.0,160.0
3,0000000015,포항영일신항만,자본잠식,-0.54,52.2,1.0,18.0
4,0000000016,일신석재,NaN,NaN,NaN,2.0,97.0
5,0000000017,일신방직,NaN,NaN,NaN,2.0,572.0
6,0000000018,매일신문사,NaN,NaN,NaN,1.0,231.0
7,0000000020,지에스정밀,NaN,NaN,NaN,1.0,1.0
8,0000000021,한화시스템,NaN,NaN,NaN,1.0,4699.0
9,0000000022,경남제일신용협동조합,NaN,NaN,NaN,1.0,NaN


In [19]:
import json

def audit_non_financial_quality(rep, emp, link):
    print("🔬 [Step 1] 비재무 데이터 품질 정밀 진단")
    print("-" * 65)

    # 1. 대표자 변경 횟수 분포 확인
    def count_ceo(info):
        try:
            data = json.loads(info)
            # '대표'라는 키워드가 들어간 직함만 카운트
            return len([x for x in data if '대표' in x.get('title', '')])
        except: return 0

    rep['CEO_Change_Count'] = rep['REPRESENTATIVE_HISTORY_INFO'].apply(count_ceo)
    print(f"📊 대표이사 변경 횟수 분포:\n{rep['CEO_Change_Count'].value_counts().sort_index()}")

    # 2. 인력 데이터 0명/결측치 비중 점검
    zero_emp = (emp['EMPLOYEE_COUNT'] == 0).sum()
    null_emp = emp['EMPLOYEE_COUNT'].isnull().sum()
    print(f"\n📊 인력 데이터 상태: 0명({zero_emp}건), 결측치({null_emp}건) / 전체({len(emp)}건)")

    # 3. 링크 데이터 카테고리별 비중
    print(f"\n📊 네트워크 카테고리 분포:\n{link['CATEGORY'].value_counts()}")

# 실행
audit_non_financial_quality(rep, emp, link)

🔬 [Step 1] 비재무 데이터 품질 정밀 진단
-----------------------------------------------------------------
📊 대표이사 변경 횟수 분포:
CEO_Change_Count
0     515
1    1053
2     160
3       7
Name: count, dtype: int64

📊 인력 데이터 상태: 0명(170건), 결측치(0건) / 전체(7881건)

📊 네트워크 카테고리 분포:
CATEGORY
payable       12519
receivable    12517
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import os

# 1. 환경 설정 및 데이터 로드
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data"
path_point = os.path.join(base_path, "output", "PROD_FLOWPOINT", "PUBLIC")

def run_network_deep_audit():
    print("🚚 [Step 1] 네트워크 원본 데이터 로드 및 ID 정규화...")
    try:
        # 원본 로드 (NameError 방지를 위해 직접 로드)
        link_raw = pd.read_csv(os.path.join(path_point, "COMPANY_LINKS.csv"), encoding='utf-8-sig')
        
        # ID 정규화 (10자리 문자열)
        link_raw['회사 아이디'] = link_raw['COMPANY_ID'].astype(str).str.split('.').str[0].str.zfill(10)
        link_raw['상대 아이디'] = link_raw['ANOTHER_COMPANY_ID'].astype(str).str.split('.').str[0].str.zfill(10)
        
        # 금액 데이터 수치화
        link_raw['AMOUNT'] = pd.to_numeric(link_raw['AMOUNT'], errors='coerce').fillna(0)
        
        print(f"✅ 데이터 로드 완료: {len(link_raw)}건의 거래 링크 확보")

        # ---------------------------------------------------------
        # [Step 2] 네트워크 질적 평가 (Deep Analysis)
        # ---------------------------------------------------------
        print("🔬 [Step 2] 네트워크 실질 지표 산출 시작...")
        
        # 기업별/카테고리별 합산
        # Receivable(채권), Payable(채무)
        pivot_df = link_raw.groupby(['회사 아이디', 'CATEGORY']).agg(
            Total_Amt=('AMOUNT', 'sum'),
            Partner_Count=('상대 아이디', 'nunique'),
            Max_Amt=('AMOUNT', 'max')
        ).unstack()
        
        # 멀티인덱스 컬럼 정리
        pivot_df.columns = [f"{c[1]}_{c[0]}" for c in pivot_df.columns]
        pivot_df = pivot_df.fillna(0)

        # 핵심 지표 1: 거래처 집중도 (Concentration Index)
        # 가장 큰 거래처 하나가 전체 거래액에서 차지하는 비중
        pivot_df['Receivable_Concentration'] = (
            pivot_df['receivable_Max_Amt'] / pivot_df['receivable_Total_Amt']
        ).replace([np.inf, -np.inf], 0).fillna(0)

        # 핵심 지표 2: 채권/채무 밸런스 (RP Ratio)
        # 받을 돈 / 줄 돈의 비율
        pivot_df['RP_Ratio'] = (
            pivot_df['receivable_Total_Amt'] / pivot_df['payable_Total_Amt']
        ).replace([np.inf, -np.inf], 0).fillna(0)

        # 핵심 지표 3: 네트워크 안정성 점수
        # 거래처가 5개 이상이고 집중도가 50% 미만이면 '우량'
        pivot_df['Network_Stability_Score'] = np.where(
            (pivot_df['receivable_Partner_Count'] >= 5) & (pivot_df['Receivable_Concentration'] < 0.5),
            20, 10
        )

        print(f"✅ 분석 완료: {len(pivot_df)}개 기업의 네트워크 프로필 생성")
        return pivot_df

    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        return pd.DataFrame()

# 최종 실행
df_network_profile = run_network_deep_audit()

# 결과 점검 (상위 10개)
if not df_network_profile.empty:
    display(df_network_profile[['receivable_Total_Amt', 'receivable_Partner_Count', 
                                'Receivable_Concentration', 'RP_Ratio', 'Network_Stability_Score']].head(10))

🚚 [Step 1] 네트워크 원본 데이터 로드 및 ID 정규화...
✅ 데이터 로드 완료: 25036건의 거래 링크 확보
🔬 [Step 2] 네트워크 실질 지표 산출 시작...
✅ 분석 완료: 874개 기업의 네트워크 프로필 생성


,receivable_Total_Amt,receivable_Partner_Count,Receivable_Concentration,RP_Ratio,Network_Stability_Score
회사 아이디,,,,,
0000000001,1.258500e+09,1.0,0.465435,0.0,10
0000000002,5.348530e+07,1.0,1.000000,0.0,10
0000000003,1.533000e+07,1.0,1.000000,0.0,10
0000000004,0.000000e+00,0.0,0.000000,0.0,10
0000000005,3.883880e+08,1.0,1.000000,0.0,10
0000000006,0.000000e+00,0.0,0.000000,0.0,10
0000000007,0.000000e+00,0.0,0.000000,0.0,10
0000000008,5.554244e+07,1.0,0.546438,0.0,10
0000000009,4.785000e+07,1.0,0.126437,0.0,10


In [ ]:
import pandas as pd
import os

# 1. BNPL 경로 설정 (방금 확인해주신 경로)
bnpl_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC\VIEW_PAY_BNPL_REQUEST_HISTORY.csv"

def audit_real_bnpl_data(path):
    print("🔬 [Real Data Audit] BNPL 요청 이력 정밀 검수...")
    print("-" * 65)
    
    try:
        df_bnpl = pd.read_csv(path, encoding='utf-8-sig')
        
        # 컬럼명 리스트 출력 (이게 모델링의 나침반입니다)
        print(f"📋 실제 컬럼 목록:\n{df_bnpl.columns.tolist()}\n")
        
        # 데이터 샘플 확인
        print("📊 데이터 샘플 (상위 3건):")
        display(df_bnpl.head(3))
        
        # 주요 상태값 분포 확인
        # 컬럼명에 'STATUS'가 포함된 것을 찾아 통계 내기
        status_col = [c for c in df_bnpl.columns if 'STATUS' in c.upper()]
        if status_col:
            print(f"\n📊 요청 상태 분포 ({status_col[0]}):")
            print(df_bnpl[status_col[0]].value_counts())
            
        return df_bnpl
    except Exception as e:
        print(f"❌ 파일 로드 실패: {e}")
        return None

# 실행
df_bnpl_raw = audit_real_bnpl_data(bnpl_path)

🔬 [Real Data Audit] BNPL 요청 이력 정밀 검수...
-----------------------------------------------------------------
📋 실제 컬럼 목록:
['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'TYPE', 'IS_READ', 'ITEM_ID', 'COMPANY_ID', 'CREATED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'IS_INTERNAL', 'COMMENT_UUID', 'ADMIN_USER_ID', 'COMMENT_CONTENT', 'EVALUATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS', 'PAY_BNPL_REQUEST_ID']

📊 데이터 샘플 (상위 3건):


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,TYPE,IS_READ,ITEM_ID,COMPANY_ID,CREATED_AT,UPDATED_AT,...,IS_INTERNAL,COMMENT_UUID,ADMIN_USER_ID,COMMENT_CONTENT,EVALUATION_STAGE,EVALUATION_STATUS,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,APPLICATION_STATUS,PAY_BNPL_REQUEST_ID
0,025182d9-252c-4316-9be2-e77c74356607,2026-02-24 14:11:47.278000+00:00,"{\n ""changes"": [],\n ""sync_id"": 788\n}",136,EVALUATION_LOG,NaN,324,NaN,2025-08-26 09:52:24.052000,NaN,...,NaN,NaN,5.0,NaN,NEGOTIATING,COMPLETE_EVALUATION,NaN,NaN,NaN,304
1,f8a150ee-fbf4-41c0-bb0e-3f2564a99864,2026-02-24 14:11:47.278000+00:00,"{\n ""changes"": [],\n ""sync_id"": 788\n}",136,EVALUATION_LOG,NaN,468,NaN,2025-11-28 09:30:31.952000,NaN,...,NaN,NaN,NaN,NaN,REVIEWING_DOCUMENTS,EVALUATING,NaN,NaN,COMPLETED,424
2,66616968-6df2-4429-8c98-9e2110ff1714,2026-02-24 14:11:47.278000+00:00,"{\n ""changes"": [],\n ""sync_id"": 788\n}",136,COMMENT,True,10,1049.0,2024-03-28 16:36:23.135000,2024-03-28 16:36:23.135000,...,False,930c5251-22f7-426b-9b9e-d2593c2cd8a6,7.0,내부심사 결과*2023년 제무제표 미제출등\n보증보험 미가입,REJECT,COMPLETE_EVALUATION,NaN,NaN,NaN,70



📊 요청 상태 분포 (EVALUATION_STATUS):
EVALUATION_STATUS
EVALUATING             266
COMPLETE_EVALUATION    172
PAYMENT                 64
COMPLETE_TRADE          57
ONGOING_SERVICE         12
Name: count, dtype: int64


In [ ]:
import pandas as pd
import os

# 1. 경로 설정
base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC"

def scan_bnpl_ecosystem():
    target_files = {
        'Requests': 'PAY_BNPL_REQUESTS.csv',
        'Buyers': 'BNPL_BUYERS.csv',
        'Sellers': 'BNPL_SELLERS.csv',
        'Items': 'PAY_BNPL_BUY_ITEMS.csv',
        'Logs': 'PAY_BNPL_REQUEST_EVALUATION_LOGS.csv'
    }
    
    print("🔬 [BNPL Ecosystem Scan] 주요 데이터셋 동시 분석 시작...")
    print("-" * 75)
    
    for label, file_name in target_files.items():
        path = os.path.join(base_path, file_name)
        if os.path.exists(path):
            df = pd.read_csv(path, encoding='utf-8-sig', nrows=5) # 샘플 5행만 로드
            print(f"\n📂 [{label}] 파일: {file_name}")
            print(f"📊 컬럼 목록: {df.columns.tolist()}")
            display(df.head(2))
        else:
            print(f"\n❌ [{label}] 파일 없음: {file_name}")

# 실행
scan_bnpl_ecosystem()

🔬 [BNPL Ecosystem Scan] 주요 데이터셋 동시 분석 시작...
---------------------------------------------------------------------------

📂 [Requests] 파일: PAY_BNPL_REQUESTS.csv
📊 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'UUID', 'USER_ID', 'PAY_TYPE', 'COMPANY_ID', 'CREATED_AT', 'DELETED_AT', 'ITEM_INDEX', 'UPDATED_AT', '_AB_CDC_LSN', 'SOURCE_TYPE', 'BUY_ITEM_NAME', 'REQUEST_NUMBER', 'TOTAL_SELL_VAT', 'TOTAL_APPLY_VAT', 'EVALUATION_STAGE', 'APPLICATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS', 'BUYER_COMPANY_NAME', 'APPLIED_COMPLETED_AT', 'TOTAL_NET_SELL_AMOUNT', 'TOTAL_NET_APPLY_AMOUNT', 'TOTAL_GROSS_SELL_AMOUNT', 'TOTAL_GROSS_APPLY_AMOUNT', 'BUYER_BUSINESS_REGISTRATION_NUMBER']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,UUID,USER_ID,PAY_TYPE,COMPANY_ID,CREATED_AT,...,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,APPLICATION_STATUS,BUYER_COMPANY_NAME,APPLIED_COMPLETED_AT,TOTAL_NET_SELL_AMOUNT,TOTAL_NET_APPLY_AMOUNT,TOTAL_GROSS_SELL_AMOUNT,TOTAL_GROSS_APPLY_AMOUNT,BUYER_BUSINESS_REGISTRATION_NUMBER
0,d80c36ac-197f-4fa9-8cc2-beeb076e5878,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,4,c5d5b5a3-56e9-4925-b025-60712b7b47dc,974,NaN,367,2025-05-07 16:49:46.618000,...,NaN,2025-10-18T02:48:35.772833664Z,CANCELLED,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16737134-0b6b-419c-ab99-15db74bd5ae8,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,19,a30fbfde-4fc0-4e0c-b3dd-e6286eb1ae5d,6,NaN,367,2025-05-20 20:11:37.076000,...,NaN,2025-10-18T02:48:35.772833664Z,CANCELLED,NaN,NaN,NaN,NaN,NaN,NaN,NaN



📂 [Buyers] 파일: BNPL_BUYERS.csv
📊 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'USER_ID', 'BANK_CODE', 'ITEM_DATE', 'ITEM_NAME', 'PAID_DATE', 'APPLY_DATE', 'CREATED_AT', 'CYCLE_UNIT', 'DELETED_AT', 'ITEM_COUNT', 'UPDATED_AT', '_AB_CDC_LSN', 'ITEM_AMOUNT', 'PAID_AMOUNT', 'APPLY_AMOUNT', 'APPROVE_DATE', 'COMPANY_NAME', 'ACCOUNT_OWNER', 'TRADE_HISTORY', 'ACCOUNT_NUMBER', 'APPROVE_AMOUNT', 'BNPL_REQUEST_ID', 'COMMISSION_RATE', 'IS_INCLUDED_VAT', 'ITEM_COUNT_UNIT', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'BUYER_MANAGER_POSITION', 'BUSINESS_REGISTRATION_NUMBER']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,USER_ID,BANK_CODE,ITEM_DATE,ITEM_NAME,PAID_DATE,...,ACCOUNT_NUMBER,APPROVE_AMOUNT,BNPL_REQUEST_ID,COMMISSION_RATE,IS_INCLUDED_VAT,ITEM_COUNT_UNIT,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,BUYER_MANAGER_POSITION,BUSINESS_REGISTRATION_NUMBER
0,fdb69c97-6d4c-477e-9c5b-6282797ec7d6,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,1,6,NaN,2024-03-13,테스트,NaN,...,NaN,NaN,1,NaN,NaN,ea,NaN,2025-10-18T02:48:35.772833664Z,NaN,12312123
1,08a8f6e4-c456-415a-aa95-2ae60ca7ce9e,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,2,6,NaN,2024-03-27,테스트,NaN,...,NaN,NaN,3,NaN,NaN,ea,NaN,2025-10-18T02:48:35.772833664Z,NaN,12312123



📂 [Sellers] 파일: BNPL_SELLERS.csv
📊 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'USER_ID', 'ITEM_DATE', 'ITEM_NAME', 'CREATED_AT', 'DELETED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'ITEM_AMOUNT', 'COMPANY_NAME', 'BNPL_REQUEST_ID', 'PAYMENT_DUE_DATE', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'BUSINESS_REGISTRATION_NUMBER']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,USER_ID,ITEM_DATE,ITEM_NAME,CREATED_AT,DELETED_AT,UPDATED_AT,_AB_CDC_LSN,ITEM_AMOUNT,COMPANY_NAME,BNPL_REQUEST_ID,PAYMENT_DUE_DATE,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,BUSINESS_REGISTRATION_NUMBER
0,20fdbf6c-111f-4cf5-9760-6ad06fca0043,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,1,6,2024-03-29,테스트2,2024-03-13 18:36:03.040000,NaN,2024-03-13 18:36:03.040000,2.869635e+13,2000000,NaN,1,NaN,NaN,2025-10-18T02:48:35.772833664Z,31482011
1,bc829a92-c607-4ec3-b0ea-a29d5a28886a,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,2,6,2024-03-13,테스트2,2024-03-13 18:41:58.757000,NaN,2024-03-13 18:41:58.757000,2.869635e+13,2000000,NaN,3,NaN,NaN,2025-10-18T02:48:35.772833664Z,11112312



📂 [Items] 파일: PAY_BNPL_BUY_ITEMS.csv
📊 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'CNT', 'NAME', 'UNIT', 'UUID', 'PRICE', 'IS_VAT', 'CATEGORY', 'TAX_TYPE', 'APPLY_VAT', 'CREATED_AT', 'DELETED_AT', 'UPDATED_AT', '_AB_CDC_LSN', 'BUY_DUE_DATE', 'PURCHASE_CYCLE', 'NET_APPLY_AMOUNT', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'GROSS_APPLY_AMOUNT', 'PAY_BNPL_COMPANY_ID', 'PAY_BNPL_REQUEST_ID']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,CNT,NAME,UNIT,UUID,PRICE,...,UPDATED_AT,_AB_CDC_LSN,BUY_DUE_DATE,PURCHASE_CYCLE,NET_APPLY_AMOUNT,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,GROSS_APPLY_AMOUNT,PAY_BNPL_COMPANY_ID,PAY_BNPL_REQUEST_ID
0,60c7f5b7-f4d8-4ab2-8601-a89ddeac24c4,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,205,NaN,NaN,NaN,c959d617-334a-4a13-816f-e98921598f27,NaN,...,2025-06-27 15:35:51.655000,2.869635e+13,NaN,NaN,NaN,NaN,2025-10-18T02:48:35.772833664Z,NaN,NaN,252
1,ecc3907c-5909-4d10-9a10-b706c9ef3fbf,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,224,NaN,NaN,NaN,3f81e048-1063-4da1-8adb-ce1175d78ac1,NaN,...,2025-07-16 11:20:39.213000,2.869635e+13,NaN,NaN,NaN,NaN,2025-10-18T02:48:35.772833664Z,NaN,NaN,276



📂 [Logs] 파일: PAY_BNPL_REQUEST_EVALUATION_LOGS.csv
📊 컬럼 목록: ['_AIRBYTE_RAW_ID', '_AIRBYTE_EXTRACTED_AT', '_AIRBYTE_META', '_AIRBYTE_GENERATION_ID', 'ID', 'CREATED_AT', 'DELETED_AT', '_AB_CDC_LSN', 'ADMIN_USER_ID', 'EVALUATION_STAGE', 'EVALUATION_STATUS', '_AB_CDC_DELETED_AT', '_AB_CDC_UPDATED_AT', 'APPLICATION_STATUS', 'PAY_BNPL_REQUEST_ID']


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,CREATED_AT,DELETED_AT,_AB_CDC_LSN,ADMIN_USER_ID,EVALUATION_STAGE,EVALUATION_STATUS,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,APPLICATION_STATUS,PAY_BNPL_REQUEST_ID
0,9263a0d7-65fe-4768-b153-13b4daec51fa,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,1,2025-04-30 17:43:03.173000,NaN,2.869635e+13,NaN,REVIEWING_DOCUMENTS,EVALUATING,NaN,2025-10-18T02:48:35.772833664Z,COMPLETED,1
1,d337308e-d458-48e9-960d-1c875f538d30,2025-10-18 02:48:35.772000+00:00,"{\n ""changes"": [],\n ""sync_id"": 24\n}",0,2,2025-05-07 17:48:02.073000,NaN,2.869635e+13,NaN,REVIEWING_DOCUMENTS,EVALUATING,NaN,2025-10-18T02:48:35.772833664Z,COMPLETED,5


In [ ]:
import pandas as pd
import numpy as np
import os

# 1. 경로 설정 (확인된 경로 기준)
base_path_point = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\PROD_FLOWPOINT\PUBLIC"

def run_bnpl_integrity_check():
    print("🚀 [Step 4] BNPL 데이터 정합성 스트레스 테스트 시작...")
    print("-" * 75)
    
    try:
        # 데이터 로드
        req = pd.read_csv(os.path.join(base_path_point, "PAY_BNPL_REQUESTS.csv"), encoding='utf-8-sig')
        items = pd.read_csv(os.path.join(base_path_point, "PAY_BNPL_BUY_ITEMS.csv"), encoding='utf-8-sig')
        buyers = pd.read_csv(os.path.join(base_path_point, "BNPL_BUYERS.csv"), encoding='utf-8-sig')
        
        # [A] ID 매칭률 (Requests ↔ Items)
        req_ids = set(req['ID'].unique())
        item_req_ids = set(items['PAY_BNPL_REQUEST_ID'].unique())
        
        match_count = len(req_ids.intersection(item_req_ids))
        match_rate = (match_count / len(req_ids)) * 100
        
        # [B] 금액 일관성 (Header vs Details)
        item_sum = items.groupby('PAY_BNPL_REQUEST_ID')['GROSS_APPLY_AMOUNT'].sum().reset_index()
        merge_check = pd.merge(req[['ID', 'TOTAL_GROSS_APPLY_AMOUNT']], item_sum, left_on='ID', right_on='PAY_BNPL_REQUEST_ID')
        merge_check['DIFF'] = abs(merge_check['TOTAL_GROSS_APPLY_AMOUNT'] - merge_check['GROSS_APPLY_AMOUNT'])
        
        error_cases = merge_check[merge_check['DIFF'] > 10] # 10원 이상 오차 발생 시 에러로 간주

        # [C] 결과 리포트 출력
        print(f"📊 1. ID 매칭률: {match_rate:.2f}% ({match_count}/{len(req_ids)} 건)")
        print(f"   - 의미: 신청서 중 상세 품목 내역이 존재하는 비중입니다.")
        
        print(f"\n📊 2. 금액 일관성: {len(error_cases)} 건의 오차 발견")
        if len(error_cases) > 0:
            print("   ⚠️ 주의: 헤더 금액과 상세 항목 합계가 다른 건이 존재합니다. (데이터 가공 필요)")
            display(error_cases.head(5))
        else:
            print("   ✅ 양호: 모든 신청 금액이 상세 항목 합계와 일치합니다.")

        print(f"\n📊 3. 데이터 밀도: 바이어 정보 {len(buyers)}건 확보됨")
        
        return merge_check
        
    except Exception as e:
        print(f"❌ 검수 중 오류 발생: {e}")
        return None

# 실행
audit_results = run_bnpl_integrity_check()

🚀 [Step 4] BNPL 데이터 정합성 스트레스 테스트 시작...
---------------------------------------------------------------------------
📊 1. ID 매칭률: 74.14% (367/495 건)
   - 의미: 신청서 중 상세 품목 내역이 존재하는 비중입니다.

📊 2. 금액 일관성: 6 건의 오차 발견
   ⚠️ 주의: 헤더 금액과 상세 항목 합계가 다른 건이 존재합니다. (데이터 가공 필요)


,ID,TOTAL_GROSS_APPLY_AMOUNT,PAY_BNPL_REQUEST_ID,GROSS_APPLY_AMOUNT,DIFF
312,457,0.0,457,2000000.0,2000000.0
314,461,0.0,461,60240000.0,60240000.0
320,464,0.0,464,50000000.0,50000000.0
342,476,10048808.0,476,19111708.0,9062900.0
347,483,0.0,483,12345.0,12345.0



📊 3. 데이터 밀도: 바이어 정보 133건 확보됨


In [ ]:
import pandas as pd
import os

base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\output\PROD_FLOWPOINT\PUBLIC"

def bnpl_full_inventory_scan():
    # BNPL_ 및 PAY_BNPL_ 로 시작하는 모든 파일 수집
    all_files = os.listdir(base_path)
    bnpl_targets = [f for f in all_files if f.startswith('BNPL_') or f.startswith('PAY_BNPL_')]
    
    print(f"📊 [Inventory Report] 총 {len(bnpl_targets)}개의 BNPL 원천 파일 발견")
    print("-" * 80)
    
    inventory = []
    for file_name in bnpl_targets:
        try:
            # 상위 1행만 읽어서 컬럼과 행수 파악 (속도 최적화)
            df_tmp = pd.read_csv(os.path.join(base_path, file_name), encoding='utf-8-sig', nrows=0)
            row_count = sum(1 for line in open(os.path.join(base_path, file_name), encoding='utf-8-sig')) - 1
            
            inventory.append({
                '파일명': file_name,
                '데이터건수': row_count,
                '컬럼수': len(df_tmp.columns),
                '주요컬럼': ", ".join(df_tmp.columns[:3].tolist()) + "..."
            })
        except:
            continue
            
    res_df = pd.DataFrame(inventory).sort_values(by='데이터건수', ascending=False)
    display(res_df)
    return res_df

# 실행
df_inventory = bnpl_full_inventory_scan()

📊 [Inventory Report] 총 26개의 BNPL 원천 파일 발견
--------------------------------------------------------------------------------


,파일명,데이터건수,컬럼수,주요컬럼
25,PAY_BNPL_UPLOAD_FILES.csv,15720,25,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
24,PAY_BNPL_TERMS.csv,6916,17,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
6,BNPL_REQUEST_APPLICATION_STAGE_LOGS.csv,4252,12,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
20,PAY_BNPL_REQUEST_PAY_BNPL_MANAGERS.csv,3104,14,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
19,PAY_BNPL_REQUEST_PAY_BNPL_COMPANIES.csv,2988,14,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
16,PAY_BNPL_MANAGERS.csv,2472,16,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
12,PAY_BNPL_COMPANIES.csv,2360,18,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
18,PAY_BNPL_REQUEST_EVALUATION_LOGS.csv,2312,15,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
4,BNPL_CUSTOMER_UPLOAD_FILES.csv,2256,19,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."
17,PAY_BNPL_REQUESTS.csv,1980,32,"_AIRBYTE_RAW_ID, _AIRBYTE_EXTRACTED_AT, _AIRBY..."


In [ ]:
import pandas as pd
import os

base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\output\PROD_FLOWPOINT\PUBLIC"

def deep_dive_hidden_signals():
    print("🔬 [Deep-Dive] BNPL 숨겨진 리스크 시그널 수사...")
    print("-" * 75)
    
    # 1. 심사 코멘트 분석 (가장 정성적인 데이터)
    try:
        comments = pd.read_csv(os.path.join(base_path, "PAY_BNPL_COMMENTS.csv"), encoding='utf-8-sig')
        print(f"\n💬 [PAY_BNPL_COMMENTS] 샘플 (심사역의 판단 근거):")
        # 실제 코멘트 내용이 들어있을 것으로 추정되는 컬럼 확인 (COMMENT, CONTENT 등)
        content_col = [c for c in comments.columns if 'CONTENT' in c.upper() or 'COMMENT' in c.upper()][0]
        display(comments[[content_col]].dropna().head(10))
    except: print("❌ 코멘트 파일 분석 실패")

    # 2. 업로드 파일 패턴 분석 (증빙의 충실도)
    try:
        files = pd.read_csv(os.path.join(base_path, "PAY_BNPL_UPLOAD_FILES.csv"), encoding='utf-8-sig')
        file_stats = files.groupby('PAY_BNPL_REQUEST_ID').size().describe()
        print(f"\n📄 [PAY_BNPL_UPLOAD_FILES] 신청당 증빙 서류 통계:")
        print(file_stats)
    except: print("❌ 업로드 파일 분석 실패")

    # 3. B2B 셀러 퀄리티 (거래 상대방의 격)
    try:
        sellers_b2b = pd.read_csv(os.path.join(base_path, "PAY_BNPL_SELLER_COMPANY_INFOS_B2B.csv"), encoding='utf-8-sig')
        print(f"\n🏢 [B2B_SELLER_INFOS] 주요 공급사 명단:")
        name_col = [c for c in sellers_b2b.columns if 'NAME' in c.upper() or 'NM' in c.upper()][0]
        display(sellers_b2b[name_col].value_counts().head(10))
    except: print("❌ B2B 셀러 정보 분석 실패")

# 실행
deep_dive_hidden_signals()

🔬 [Deep-Dive] BNPL 숨겨진 리스크 시그널 수사...
---------------------------------------------------------------------------

💬 [PAY_BNPL_COMMENTS] 샘플 (심사역의 판단 근거):


,CONTENT
0,거절합니다.
1,고객사의 재무상태와 각종 이슈로 진행이 어려움
2,브라질 조류독감으로 인한 수입중단 발생\n
3,브라질 조류독감으로 인한 냉동닭 수입 중단
4,협업구조의 각각의 사업자 재무상태 및 거래상태 불량
5,다음의 몇가지 사항으로 최총 투자매칭이 부결되어 안내드립니다.\n1. 매출감소 및 ...
6,"1. 서비스 제공 조건\n 구매액 : 45,000,000원(VAT포함)\n ..."
7,1. 판매처와 관련된 발주서 또는 계약서를 통해 실제 거래내역을 확인하고자 합니다....
8,견적서 재요청/수정완료
9,내부심사 결과*2023년 제무제표 미제출등\n보증보험 미가입



📄 [PAY_BNPL_UPLOAD_FILES] 신청당 증빙 서류 통계:
count    297.000000
mean      13.232323
std        8.226722
min        1.000000
25%        8.000000
50%       12.000000
75%       18.000000
max       45.000000
dtype: float64

🏢 [B2B_SELLER_INFOS] 주요 공급사 명단:
❌ B2B 셀러 정보 분석 실패


In [ ]:
import pandas as pd
import os

base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\output\PROD_FLOWPOINT\PUBLIC"

def analyze_transaction_relations():
    print("🕸️ [Jacob's Deep-Scan] 거래 관계 및 계약 상태 정밀 분석")
    print("-" * 80)

    # 1. Buyer-Seller 매핑 (거래 생태계 복구)
    try:
        buyers = pd.read_csv(os.path.join(base_path, "BNPL_BUYERS.csv"), encoding='utf-8-sig')
        sellers = pd.read_csv(os.path.join(base_path, "BNPL_SELLERS.csv"), encoding='utf-8-sig')
        
        # BNPL_REQUEST_ID 기준으로 매핑
        network = pd.merge(
            buyers[['BNPL_REQUEST_ID', 'COMPANY_NAME', 'ITEM_NAME']].rename(columns={'COMPANY_NAME': '구매자', 'ITEM_NAME': '구매품목'}),
            sellers[['BNPL_REQUEST_ID', 'COMPANY_NAME']].rename(columns={'COMPANY_NAME': '판매자'}),
            on='BNPL_REQUEST_ID', how='inner'
        )
        print(f"🔗 [Network] 주요 거래 관계 (Top 10):\n")
        display(network.groupby(['구매자', '판매자']).size().sort_values(ascending=False).head(10))
    except: print("❌ Buyer-Seller 매핑 실패")

    # 2. 계약 조건(Terms) 상태 분석
    try:
        terms = pd.read_csv(os.path.join(base_path, "PAY_BNPL_TERMS.csv"), encoding='utf-8-sig')
        # 약관 동의 상태나 유형 컬럼이 있는지 확인 (예: STATUS, TYPE)
        status_col = [c for c in terms.columns if 'STATUS' in c.upper() or 'TYPE' in c.upper()][0]
        print(f"\n📑 [Terms] 계약 조건 분포 ({status_col}):")
        print(terms[status_col].value_counts().head(10))
    except: print("❌ Terms 파일 분석 실패")

    # 3. 비즈니스 관계 (Business Relations)
    try:
        biz_rel = pd.read_csv(os.path.join(base_path, "BUSINESS_RELATIONS.csv"), encoding='utf-8-sig')
        print(f"\n🏢 [Business Relations] 명시적 기업 관계:")
        display(biz_rel.head(5))
    except: print("❌ BUSINESS_RELATIONS 파일 없음 (건너뜀)")

# 실행
analyze_transaction_relations()

🕸️ [Jacob's Deep-Scan] 거래 관계 및 계약 상태 정밀 분석
--------------------------------------------------------------------------------
🔗 [Network] 주요 거래 관계 (Top 10):



Series([], dtype: int64)

❌ Terms 파일 분석 실패

🏢 [Business Relations] 명시적 기업 관계:


,_AIRBYTE_RAW_ID,_AIRBYTE_EXTRACTED_AT,_AIRBYTE_META,_AIRBYTE_GENERATION_ID,ID,IS_MAIN,USER_ID,CREATED_AT,DELETED_AT,UPDATED_AT,...,_AB_CDC_DELETED_AT,_AB_CDC_UPDATED_AT,COMPANY_FAX_NUMBER,MANAGER_DEPARTMENT,COMPANY_PHONE_NUMBER,MANAGER_PHONE_NUMBER,COMPANY_BUSINESS_TYPE,COMPANY_DETAIL_ADDRESS,COMPANY_ESTABLISHMENT_DATE,COMPANY_REGISTRATION_NUMBER


In [ ]:
import pandas as pd
import os

base_path = r"C:\Users\cozy1\Documents\276_Scoring_Model\raw_data\output\PROD_FLOWPOINT\PUBLIC"

def mini_data_spec_audit():
    print("🔍 [Mini-Test] 조인 키(Join Key) 데이터 규격 감사")
    print("-" * 70)
    
    files_to_test = {
        'Buyers': 'BNPL_BUYERS.csv',
        'Sellers': 'BNPL_SELLERS.csv',
        'Terms': 'PAY_BNPL_TERMS.csv',
        'Requests': 'PAY_BNPL_REQUESTS.csv'
    }
    
    spec_report = []
    
    for label, file_name in files_to_test.items():
        path = os.path.join(base_path, file_name)
        if os.path.exists(path):
            # 샘플 5개만 로드
            df = pd.read_csv(path, encoding='utf-8-sig', nrows=5)
            
            # ID 컬럼 찾기 (BNPL_REQUEST_ID 혹은 ID 등)
            id_col = [c for c in df.columns if 'REQUEST_ID' in c.upper() or c == 'ID'][0]
            
            sample_val = df[id_col].iloc[0]
            spec_report.append({
                '파일': label,
                '대상컬럼': id_col,
                '데이터타입': df[id_col].dtype,
                '샘플값': sample_val,
                '타입(Python)': type(sample_val).__name__
            })
            
    res_df = pd.DataFrame(spec_report)
    display(res_df)
    
    # 타입 불일치 경고
    if len(res_df['데이터타입'].unique()) > 1:
        print("\n⚠️ [경고] 파일 간 ID 데이터 타입이 일치하지 않습니다. 강제 형변환이 필요합니다.")
    else:
        print("\n✅ [통과] 데이터 타입이 일치합니다.")

# 실행
mini_data_spec_audit()

🔍 [Mini-Test] 조인 키(Join Key) 데이터 규격 감사
----------------------------------------------------------------------


,파일,대상컬럼,데이터타입,샘플값,타입(Python)
0,Buyers,ID,int64,1,int64
1,Sellers,ID,int64,1,int64
2,Terms,ID,int64,5,int64
3,Requests,ID,int64,4,int64



✅ [통과] 데이터 타입이 일치합니다.


In [34]:
# [Jacob's Unit Test 2] 실제 연결 키 정합성 확인
def verify_linkage_keys():
    print("🔗 [Linkage-Test] 실제 조인 키 데이터 규격 확인")
    print("-" * 70)
    
    # 우리가 실제로 조인할 관계 정의
    relationships = [
        {'from': 'Buyers', 'file': 'BNPL_BUYERS.csv', 'key': 'BNPL_REQUEST_ID'},
        {'from': 'Sellers', 'file': 'BNPL_SELLERS.csv', 'key': 'BNPL_REQUEST_ID'},
        {'from': 'Terms', 'file': 'PAY_BNPL_TERMS.csv', 'key': 'PAY_BNPL_REQUEST_ID'},
        {'from': 'Requests', 'file': 'PAY_BNPL_REQUESTS.csv', 'key': 'ID'}
    ]
    
    link_report = []
    for rel in relationships:
        path = os.path.join(base_path, rel['file'])
        if os.path.exists(path):
            df = pd.read_csv(path, encoding='utf-8-sig', nrows=5)
            col = rel['key']
            if col in df.columns:
                val = df[col].iloc[0]
                link_report.append({
                    '파일': rel['from'],
                    '연결키': col,
                    '타입': df[col].dtype,
                    '샘플값': val
                })
    
    display(pd.DataFrame(link_report))

verify_linkage_keys()

🔗 [Linkage-Test] 실제 조인 키 데이터 규격 확인
----------------------------------------------------------------------


,파일,연결키,타입,샘플값
0,Buyers,BNPL_REQUEST_ID,int64,1
1,Sellers,BNPL_REQUEST_ID,int64,1
2,Terms,PAY_BNPL_REQUEST_ID,int64,1
3,Requests,ID,int64,4
